# Regularisierung

In diesem Notebook untersuchen wir, wie Regularisierungsverfahren überangepasste lineare Modelle stabilisieren. Wir arbeiten mit dem **Prostate-Cancer-Datensatz** (Stamey et al., 1989), den Sie bereits aus der Vorlesung kennen, und erweitern ihn mit polynomialen Merkmalen, um eine realistisch hochdimensionale Situation zu schaffen.

### Überblick

1. **Ridge**-, **Lasso**- und **Elastic-Net**-Regression in `sklearn`.
2. Ridge-Koeffizienten in geschlossener Form mit `numpy` selbst berechnen.
3. **Koeffizientenpfade** interpretieren und L1- von L2-Verhalten unterscheiden.
4. **Effektive Freiheitsgrade** und der **Schrumpfungsfaktor**.
5. Regularisierungsparameter $\lambda$ durch **Kreuzvalidierung** wählen.
6. Typische Stolpersteine bei `sklearn`-Parametern.

## Benötigte Module

In [ ]:
from itertools import combinations
import matplotlib.pyplot as plt
import numpy as np
import os

from sklearn.linear_model import (
    LinearRegression, Ridge, Lasso, ElasticNet,
    RidgeCV, LassoCV, ElasticNetCV
)
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import KFold, train_test_split
from sklearn.preprocessing import StandardScaler, PolynomialFeatures

## Reproduzierbarkeit

In [ ]:
SEED = 19751979
generator = np.random.default_rng(SEED)

## Daten laden und vorbereiten

Wir benutzen den Prostate-Cancer-Datensatz mit 97 Beobachtungen, 8 Prädiktoren und einer kontinuierlichen Zielvariable `lpsa` (log-PSA-Wert). Die Spalten sind:

| Spalte | Bedeutung |
|---|---|
| `lcavol`  | log(Tumor-Volumen) |
| `lweight` | log(Prostata-Gewicht) |
| `age`     | Alter |
| `lbph`    | log(benigne Prostatahyperplasie) |
| `svi`     | Samenblaseninvasion (0/1) |
| `lcp`     | log(Kapselpenetration) |
| `gleason` | Gleason-Score |
| `pgg45`   | Prozent Gleason 4 oder 5 |
| `lpsa`    | **Ziel:** log([prostataspezifisches Antigen](https://de.wikipedia.org/wiki/Prostataspezifisches_Antigen)) |


In [ ]:
def load_prostate(file_dataset: str, column_names: list[str]) -> tuple[np.ndarray, np.ndarray]:
    """
    Ladet den Prostatakrebs-Datensatz von lokaler Datei.
    :param file_dataset: Pfad zur CSV-Datei.
    :param column_names: Spaltennamen, die in der Kopfzeile definiert werden sollten.
    :return: Datensatz als Tupel der Form `(x, y)`.
    """
    if not os.path.isfile(file_dataset):
        raise FileNotFoundError('file ' + file_dataset)
    x, y = [], []
    with open(file_dataset, 'r') as file_handle:
        for index_line, line in enumerate(file_handle):
            line = line.strip('\r\n')
            if index_line == 0:
                if line == '':
                    raise ValueError(f'Empty header line in {file_dataset}')
                column_names_observed = line.split(',')
                if len(column_names_observed) != len(column_names):
                    raise ValueError(f'Unexpected number of columns in {file_dataset}')
                if column_names_observed != column_names:
                    raise ValueError(f'Unexpected column names in {file_dataset}')
            elif line != '':
                error_text = ''
                try:
                    values = [float(x) for x in line.split(',')]
                    if len(values) != len(column_names):
                        error_text = f'Unexpected number of columns '
                    x.append(values[:-1])
                    y.append(values[-1])
                except ValueError:
                    error_text = f'Invalid value'
                if error_text != '':
                    raise ValueError(error_text + f' in line {index_line + 1} in {file_dataset}')
    return np.array(x, dtype=np.float64), np.array(y, dtype=np.float64)


file_dataset = 'prostate-cancer.csv'
columns = ['lcavol', 'lweight', 'age', 'lbph', 'svi', 'lcp', 'gleason', 'pgg45', 'lpsa']
x, y = load_prostate(file_dataset, columns)

### Trainings- und Testaufteilung

Da der Datensatz keine Standard-Train/Test-Spalte enthält, verwenden wir eine deterministische Aufteilung mit [`train_test_split`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html).

In [ ]:
x_train_raw, x_test_raw, y_train, y_test =\
    train_test_split(x, y, test_size=0.3, random_state=SEED)

print(f'Form des Trainingssatzes: {x_train_raw.shape}')
print(f'     Form des Testsatzes: {x_test_raw.shape}')

In [ ]:
print('Werte der Zielvariable im Trainingssatz:')
print(y[:20])
print('Werte der Zielvariable im Testsatz:')
print(y_test[:20])

### Standardisierung

Bei jeder Form der Regularisierung müssen die Merkmale auf vergleichbare Skalen gebracht werden &mdash; sonst werden Merkmale mit grossen Wertebereichen stärker bestraft als solche mit kleinen Wertebereichen. Wir verwenden [`StandardScaler`](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html) (Mittelwert 0, Standardabweichung 1).

In [ ]:
scaler = StandardScaler()
x_train = scaler.fit_transform(x_train_raw)
x_test = scaler.transform(x_test_raw)

## Teilmengenauswahl

Bevor wir uns auf Shrinkage-Methoden stürzen, ein kurzer Blick auf die explizite Variablenselektion.

### Best Subset Selection

Wir suchen für jede Grösse $k \in \{1, \dots, p\}$ die *beste* Teilmenge von $k$ Merkmalen &mdash; also das Modell mit dem kleinsten RSS. Die Anzahl zu prüfender Teilmengen ist

$$
\sum_{k=0}^{p} \binom{p}{k} = 2^p.
$$

Mit dem 8-Merkmale-Datensatz wären es $2^8 = 256$ Modelle, noch handhabbar.

Die folgende Funktion implementiert diese Methode mit der Hilfe der Funktion `combinations` vom Paket [`itertools`](https://docs.python.org/3/library/itertools.html). Einzelne lineare Regressionsmodelle werden mit [`np.linalg.lstsq`](https://numpy.org/doc/stable/reference/generated/numpy.linalg.lstsq.html) trainiert.

In [ ]:
def add_bias(x: np.ndarray) -> np.ndarray:
    """
    Fügt eine Bias-Spalte zur Merkmalsmatrix hinzu.
    :param x: Merkmalsmatrix als Array der Form `(n, p)`.
    :return: Array der Form `(n, p + 1)` in dem die erste Spalte mit Einsen gefüllt ist.
    """
    return np.column_stack((np.ones(x.shape[0], dtype=x.dtype), x))


def best_subset_selection(x: np.ndarray, y: np.ndarray) -> tuple[list, np.ndarray]:
    """
    Führt eine vollständige Teilmengenauswahl per OLS durch.
    :param x: Trainings-Designmatrix der Form `(n, p)`.
    :param y: Trainingsziel (zentriert) der Form `(n,)`.
    :return: Tupel der Form `(Features, MSEs)`.
    """
    p = x.shape[1]
    indices = list(range(p))
    best_subsets = [() for _ in range(p + 1)]
    best_mse = np.full(p + 1, np.inf, dtype=np.float64)
    best_mse[0] = np.mean((y - y.mean()) ** 2)
    for k in range(1, p + 1):
        # Über alle Teilmengen der Grösse k iterieren
        for subset in combinations(indices, k):
            x_subset = add_bias(x[:, list(subset)])
            betas, *_ = np.linalg.lstsq(x_subset, y)
            mse = np.mean((y - x_subset @ betas) ** 2)
            if mse < best_mse[k]:
                best_mse[k] = mse
                best_subsets[k] = subset
    return best_subsets, best_mse


# Auf den 8 Originalmerkmalen
best_subsets, best_mse = best_subset_selection(x_train, y_train)
print(best_subsets)

### Visualisierung des Best-Subset-Pfads


In [ ]:
def plot_best_subset_path(best_mse: np.ndarray) -> plt.Figure:
    """
    Visualisiert RSS gegen Anzahl ausgewählter Merkmale für Best Subset Selection.
    :param best_rss: als der zwiete Rückgabewert von `best_subset_selection`.
    :return: das neu erstellte `Figure`-Objekt.
    """
    ks = np.arange(best_mse.size)
    figure, ax = plt.subplots(figsize=(6, 3.5), dpi=100)
    ax.plot(ks, best_mse, 'o--', color='tab:blue')
    ax.set(xlabel='Anzahl ausgewählter Merkmale', ylabel='Trainings-MSE')
    ax.grid(color='#A0A0A0', linestyle='--', linewidth=0.5)
    figure.tight_layout()
    return figure


figure = plot_best_subset_path(best_mse)
plt.show(figure)

**Beobachtung:** Der RSS sinkt monoton mit $k$ &mdash; mehr ist immer besser auf den Trainingsdaten. Diese Ergebnisse zeigen uns nicht, welches $k$ wir wählen sollten; dafür brauchen wir ein Validierungs- oder Kreuzvalidierungs-Verfahren.

**Hauptproblem von Best Subset:** Die Methode ist kombinatorisch teuer und hohe Varianz der Selektion &mdash; eine einzelne Beobachtung mehr oder weniger kann zu einer komplett anderen Teilmenge führen. Die nachfolgenden Shrinkage-Verfahren sind kontinuierlich und deshalb stabiler.

Die in `sklearn` verfügbaren Klassen [`SequentialFeatureSelector`](https://scikit-learn.org/stable/modules/generated/sklearn.feature_selection.SequentialFeatureSelector.html) und [`RFE`](https://scikit-learn.org/stable/modules/generated/sklearn.feature_selection.RFE.html#sklearn.feature_selection.RFE) implementieren *Vorwärts*- bzw. *Rückwärts-Stepwise-Selection* &mdash; eine pragmatische Approximation an Best Subset.

> ### Achtung: Namenskonventionen in `sklearn`
>
> Bevor wir loslegen, ein wichtiger Hinweis. In der Vorlesung haben wir den Regularisierungsparameter mit $\lambda$ bezeichnet. In `sklearn` heisst dieser Parameter **`alpha`** &mdash; in `Ridge`, `Lasso` und `ElasticNet`.
>
> Verwirrend wird es bei `ElasticNet`: dort gibt es **zusätzlich** einen Parameter `l1_ratio`, der die Mischung zwischen L1 und L2 steuert (in der Vorlesung haben wir dafür den Buchstaben $\alpha$ benutzt). `sklearn` benutzt also den Namen `alpha` für zwei verschiedene Dinge in zwei verschiedenen Kontexten.
>
> Bei der `LogisticRegression` heisst der Regularisierungsparameter wiederum **`C`** und ist die inverse Stärke: $C = 1/\lambda$. Grösseres `C` bedeutet *weniger* Regularisierung.
>
> Wir kommen am Ende der Sitzung noch einmal auf diese Stolpersteine zurück.

## OLS-Baseline

Zur Erinnerung: die gewöhnliche kleinste-Quadrate-Methode (Ordinary Least Squares, OLS) minimiert

$$
\mathcal{L}_{\text{OLS}}(\boldsymbol\beta) \;=\; \|\mathbf{y} - \mathbf{X}\boldsymbol\beta\|_2^2.
$$

Wir trainieren ein OLS-Modell auf den 8 Originalmerkmalen als Vergleichspunkt. Wir berechnen die Trainings- und Testverlustwerte mit der Funktion [`mean_squared_error`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.mean_squared_error.html).

In [ ]:
def evaluate(model, x_train, x_test, y_train, y_test) -> tuple[float, float]:
    """
    Berechnet Trainings- und Test-MSE.
    :param model: ein gefittetes sklearn-Modell.
    :param x_train: Trainingsmerkmale (standardisiert).
    :param x_test: Testmerkmale (standardisiert).
    :param y_train: Trainingsziel (unzentriert).
    :param y_test: Testziel.
    :return: Train- und Test-MSE als Array der Form `(2)`.
    """
    result = np.empty(2, dtype=np.float64)
    result[0] = mean_squared_error(y_train, model.predict(x_train))
    result[1] = mean_squared_error(y_test, model.predict(x_test))
    return result


ols_baseline = LinearRegression(fit_intercept=True)
ols_baseline.fit(x_train, y_train)
mse_baseline = evaluate(ols_baseline, x_train, x_test, y_train, y_test)
print(f'OLS 8 Merkmale, Train MSE = {mse_baseline[0]:.3f}')
print(f' OLS 8 Merkmale, Test MSE = {mse_baseline[1]:.3f}')

## Merkmals-Expansion

Mit 8 Merkmalen und 67 Trainingsbeobachtungen ist OLS noch ziemlich brav. Spannend wird es, wenn wir die Merkmalszahl erhöhen &mdash; und genau das tun wir jetzt mit [`PolynomialFeatures(degree=2)`](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.PolynomialFeatures.html). Daraus entstehen aus 8 Merkmalen:

- 8 Originalmerkmale
- 8 quadratische Terme ($x_j^2$)
- $\binom{8}{2}=28$ paarweise Interaktionen ($x_j \cdot x_k$)

Insgesamt **44 Merkmale** &mdash; gegenüber 67 Trainingsbeobachtungen ein gefährliches Verhältnis.

Best Subset Selection sollte bei $p = 44$ Features insgesamt $2^{44} \approx 1{,}76 \cdot 10^{13}$ Modelle traainieren &mdash; **nicht durchführbar**.

In [ ]:
poly = PolynomialFeatures(degree=2, interaction_only=False, include_bias=False)
x_train_poly_raw = poly.fit_transform(x_train_raw)
x_test_poly_raw = poly.transform(x_test_raw)

# Erneute Standardisierung der erweiterten Merkmale
scaler_poly = StandardScaler()
x_train_poly = scaler_poly.fit_transform(x_train_poly_raw)
x_test_poly = scaler_poly.transform(x_test_poly_raw)

feature_names = columns[:-1]
feature_names_poly = poly.get_feature_names_out(feature_names)
n_features_poly = x_train_poly.shape[1]

print(f'Form der Merkmalsmatrix nach der Expansion: {x_train_poly.shape}')
print(f'Namen der Features im erweiterten Satz: {feature_names_poly}')

### Übung

Vor dem Programmieren: **was vermuten Sie?**

- Wie verändert sich der **Trainings**-MSE, wenn wir von 8 auf 44 Merkmale gehen?
- Wie verändert sich der **Test**-MSE?
- Wie sehen die Koeffizientenbeträge aus &mdash; eher klein oder eher gross?

---

**Ihre Aufgabe:**

1. Trainieren Sie ein OLS-Modell auf `x_train_poly` und `y_train`. Speichern Sie das Modell in der Variable `ols_poly`.
2. Berechnen Sie Train- und Test-MSE (verwenden Sie die `evaluate`-Funktion von oben).
3. Vergleichen Sie die Werte mit dem 8-Merkmale-Baseline-Modell.

In [ ]:
# 1. OLS auf polynomiale Merkmale trainieren
ols_poly = LinearRegression()
ols_poly.fit(x_train_poly, y_train)

# 2. MSE-Werte berechnen und ausgeben
mse_poly = evaluate(ols_poly, x_train_poly, x_test_poly, y_train, y_test)
print(f' OLS 8 Merkmale, Train MSE = {mse_baseline[0]:.3f}')
print(f'  OLS 8 Merkmale, Test MSE = {mse_baseline[1]:.3f}')
print(f'OLS 44 Merkmale, Train MSE = {mse_poly[0]:.3f}')
print(f' OLS 44 Merkmale, Test MSE = {mse_poly[1]:.3f}')

**Beobachtung:** Der Trainings-MSE ist drastisch gesunken, der Test-MSE aber deutlich gestiegen. Die Koeffizientenbeträge sind explodiert. Klassischer Fall von Überanpassung &mdash; und der perfekte Anlass für Regularisierung.

Schauen wir uns die Koeffizienten an.

In [ ]:
def plot_coefficient_bars(coefs: np.ndarray,
                          feature_names: np.ndarray,
                          sort_by_abs: bool = False) -> plt.Figure:
    """
    Visualisiert die Koeffizienten eines linearen Modells als Balkendiagramm.
    :param coefs: Koeffizientenvektor der Form `(p,)`.
    :param feature_names: Namen der Merkmale, Länge `p`.
    :param sort_by_abs: wenn `True`, werden die Balken nach Absolutbetrag sortiert.
    :return: das neu erstelle `Figure`-Objekt.
    """
    figure, ax = plt.subplots(figsize=(9, 4), dpi=100)
    coefs = np.asarray(coefs)
    if sort_by_abs:
        order = np.argsort(-np.abs(coefs))
        coefs = coefs[order]
        feature_names = np.asarray(feature_names)[order]

    p = len(coefs)
    colors = ['tab:blue' if c >= 0 else 'tab:red' for c in coefs]
    ax.bar(range(len(coefs)), coefs, color=colors)
    ax.axhline(0, color='black', linewidth=0.6)
    ax.set(axisbelow=True, xlim=[-0.5, len(coefs) - 0.5], xticks=range(p))
    ax.set_xticklabels(feature_names, rotation=90, fontsize=8)
    ax.set_ylabel(r'$\hat{\beta}_j$', rotation=0)
    ax.grid(color='#A0A0A0', linestyle='--', linewidth=0.5)
    figure.tight_layout()
    return figure


figure = plot_coefficient_bars(ols_poly.coef_, feature_names_poly)
plt.show(figure)

## Ridge-Regression (L2-Regularisierung)

Ridge minimiert die mit einem L2-Strafterm erweiterte Verlustfunktion:

$$
\hat{\boldsymbol\beta}_{\text{Ridge}}
\;=\; \arg\min_{\boldsymbol\beta}
\Big\{ \underbrace{\|\mathbf{y} - \mathbf{X}\boldsymbol\beta\|_2^2}_{\text{RSS}}
       \;+\; \lambda\,\|\boldsymbol\beta\|_2^2 \Big\}.
$$

Wegen der Differenzierbarkeit des Strafterms existiert eine **geschlossene Lösung**:

$$
\boxed{\;\hat{\boldsymbol\beta}_{\text{Ridge}}
       = \bigl(\mathbf{X}^\top \mathbf{X} + \lambda \mathbf{I}_p\bigr)^{-1}
         \mathbf{X}^\top \mathbf{y}.\;}
$$

Diese Formel ist eine simple Erweiterung der OLS-Lösung $\hat{\boldsymbol\beta}_{\text{OLS}} = (\mathbf{X}^\top \mathbf{X})^{-1} \mathbf{X}^\top \mathbf{y}$: Der Term $\lambda \mathbf{I}_p$ macht die Matrix immer invertierbar, auch wenn $\mathbf{X}^\top \mathbf{X}$ singulär ist (z. B. bei korrelierten Merkmalen oder $p > n$).


### Übung: Ridge in `numpy` implementieren

Implementieren Sie die Ridge-Lösung mit der Hilfe der Funktionalität in numpy.

1. Vervollständigen Sie die Funktion `ridge_closed_form(x, y, alpha)`.
2. Berechnen Sie die Koeffizienten für $\lambda = 1{,}0$.
3. Vergleichen Sie das Ergebnis mit `sklearn.linear_model.Ridge`.

**Hinweise:**
- Identitätsmatrix kann mit [`np.eye(N)`](https://numpy.org/doc/stable/reference/generated/numpy.eye.html#numpy.eye) initialisiert werden.
- Die Funktion [`np.linalg.solve(A, b)`](https://numpy.org/doc/stable/reference/generated/numpy.linalg.solve.html) löst dieselbe Gleichung wie `np.linalg.inv(A) @ b`, aber numerisch stabiler und schneller.

In [ ]:
def ridge_closed_form(x: np.ndarray, y: np.ndarray, alpha: float) -> np.ndarray:
    """
    Berechnet die Ridge-Regression-Koeffizienten in geschlossener Form.
    :param x: standardisierte Designmatrix der Form `(n, p)` (ohne Intercept-Spalte).
    :param y: Zielvektor der Form `(n,)`.
    :param alpha: Regularisierungsstärke lambda als nicht-negative `float` Zahl.
    :return: Koeffizientenvektor der Form `(p,)`.
    """
    A = x.T @ x + alpha * np.eye(x.shape[1])
    return np.linalg.solve(A, x.T @ y)


alpha = 1.0
beta_np = ridge_closed_form(x_train_poly, y_train, alpha=alpha)

ridge_sk = Ridge(alpha=alpha, fit_intercept=True)
ridge_sk.fit(x_train_poly, y_train)
beta_sk = ridge_sk.coef_

print(f'  Gewichte 1 bis 5 von Ridge Regression (numpy): {beta_np[:5]}')
print(f'Gewichte 1 bis 5 von Ridge Regression (sklearn): {beta_sk[:5]}')

Die Werte sollten bis auf numerische Genauigkeit übereinstimmen.

> **`sklearn`-Detail:** Für Ridge entspricht `alpha` in `sklearn` **exakt** dem $\lambda$ aus der Vorlesung. Bei Lasso und Elastic Net gibt es einen zusätzlichen Vorfaktor $\frac{1}{2n}$ vor dem Datenterm; das schauen wir uns später genauer an.

### Ridge-Koeffizientenpfad

Wir wiederholen die Berechnung jetzt für viele $\lambda$-Werte und plotten, wie sich jeder Koeffizient mit zunehmender Regularisierung verändert. Das Ergebnis &mdash; der **Koeffizientenpfad** &mdash; ist oft verwendetes Diagramm im Maschinellen Lernen.

In [ ]:
def compute_coefficient_path(
    x: np.ndarray,
    y: np.ndarray,
    lambdas: np.ndarray,
    method: str = 'ridge',
    l1_ratio: float = 0.5,
) -> np.ndarray:
    """
    Berechnet den Koeffizientenpfad für eine Sequenz von Regularisierungsstärken.
    :param x: standardisierte Designmatrix der Form `(n, p)`.
    :param y: zentriertes Zielvektor der Form `(n,)`.
    :param lambdas: 1D-Array mit Regularisierungsstärken (sortiert).
    :param method: einer von "ridge", "lasso", "elasticnet".
    :param l1_ratio: nur für "elasticnet" relevant.
    :return: Matrix der Form `(len(lambdas), p + 1)` mit den Koeffizienten.
    """
    result = np.empty((lambdas.size, x.shape[1] + 1), dtype=np.float64)
    for i, lam in enumerate(lambdas):
        if method == 'ridge':
            model = Ridge(alpha=lam)
        elif method == 'lasso':
            model = Lasso(alpha=lam, max_iter=100000)
        elif method == 'elasticnet':
            model = ElasticNet(alpha=lam, l1_ratio=l1_ratio, max_iter=100000)
        else:
            raise ValueError(f'Unbekannte Methode: {method}')
        model.fit(x, y)
        result[i, 0] = model.intercept_
        result[i, 1:] = model.coef_
    return result


def plot_coefficient_path(
    lambdas: np.ndarray,
    coefs_matrix: np.ndarray,
    feature_names: np.ndarray,
    highlight_top_k: int = 8,
) -> plt.Figure:
    """
    Visualisiert den Koeffizientenpfad gegen Lambda (logarithmisiert).
    :param lambdas: 1D-Array mit Regularisierungsstärken.
    :param coefs_matrix: Matrix der Form `(len(lambdas), p)`.
    :param feature_names: Namen der Merkmale, Länge `p`.
    :param highlight_top_k: Anzahl der zu beschriftenden, dominantesten Merkmale.
    :return: das `Figure`-Objekt.
    """
    figure, ax = plt.subplots(figsize=(10, 5), dpi=100)
    # Dominanteste Merkmale bestimmen (anhand des maximalen Absolutbetrags entlang des Pfads)
    max_abs = np.max(np.abs(coefs_matrix[:, 1:]), axis=0)
    top_idx = np.argsort(-max_abs)[:highlight_top_k]
    xs = np.log10(lambdas)
    for j in range(coefs_matrix.shape[1] - 1):
        color = None if j in top_idx else 'lightgray'
        label = feature_names[j] if j in top_idx else None
        zorder = 3 if j in top_idx else 1
        ax.plot(xs, coefs_matrix[:, j + 1],
                color=color, label=label, zorder=zorder, linewidth=1.4)
    ax.axhline(0, color='black', linewidth=0.5)
    ax.set(axisbelow=True, xlim=[xs[0], xs[-1]])
    ax.set(xlabel=r'$\log_{10}(\lambda)$', ylabel=r'Koeffizientenwert $\hat{\beta}_j$')
    ax.legend(loc='best', fontsize=8, ncol=2)
    ax.grid(color='#A0A0A0', linestyle='--', linewidth=0.5)
    figure.tight_layout()
    return figure

In [ ]:
lambdas_ridge = np.logspace(-3, 4, 176)
coefs_ridge =\
    compute_coefficient_path(x_train_poly, y_train, lambdas_ridge, method='ridge')
figure = plot_coefficient_path(lambdas_ridge, coefs_ridge, feature_names_poly)
plt.show(figure)

**Interpretation:**

- Bei kleinem $\lambda$ (links) sind die Koeffizienten so wie bei OLS &mdash; mit grossen Beträgen.
- Bei wachsendem $\lambda$ werden alle Koeffizienten gleichmässig zur Null hin geschrumpft.
- Die Pfade sind glatt und stetig.

### Effektive Freiheitsgrade von Ridge

Das gewöhnliche OLS-Modell mit $p$ Merkmalen hat **$p$ Freiheitsgrade**. Wie viele Freiheitsgrade hat ein Ridge-Modell? &mdash; Diese Frage ist überraschend tief und hat eine elegante Antwort über die [Singulärwertzerlegung](https://de.wikipedia.org/wiki/Singul%C3%A4rwertzerlegung) (SVD).

Sei $\mathbf{X} = \mathbf{U}\mathbf{D}\mathbf{V}^\top$ die SVD der standardisierten Designmatrix mit Singulärwerten $d_1 \geq d_2 \geq \dots \geq d_p \geq 0$. Dann ist die **effektive Freiheitsgradzahl** von Ridge:

$$
\operatorname{df}(\lambda) \;=\; \operatorname{tr}\bigl(\mathbf{X}(\mathbf{X}^\top \mathbf{X} + \lambda \mathbf{I})^{-1}\mathbf{X}^\top\bigr) \;=\; \sum_{j=1}^{p} \frac{d_j^2}{d_j^2 + \lambda}.
$$

**Grenzfälle:**

- $\lambda = 0$:  $\operatorname{df}(0) = \sum_j \frac{d_j^2}{d_j^2} = p$ &nbsp;&mdash;&nbsp; volle Freiheit, wie bei OLS.
- $\lambda \to \infty$:  $\operatorname{df}(\lambda) \to 0$ &nbsp;&mdash;&nbsp; das Modell wird quasi konstant.

Jeder Summand $\frac{d_j^2}{d_j^2 + \lambda} \in (0, 1)$ ist eine *Bruchteils-Freiheit* der entsprechenden Hauptkomponente.


In [ ]:
def effective_df_ridge(x: np.ndarray, lambdas: np.ndarray) -> np.ndarray:
    """
    Berechnet die effektiven Freiheitsgrade (DF) einer Ridge-Regression.
    :param x: standardisierte Merkmalsmatrix der Form `(n, p)`.
    :param lambdas: 1D-Array mit Regularisierungsstärken.
    :return: 1D-Array der Form `(lambdas.size,)` mit den DF-Werten.
    """
    s = np.linalg.svd(x, compute_uv=False)  # Singulärwerte d_j
    d2 = s ** 2
    return np.array([np.sum(d2 / (d2 + lam)) for lam in lambdas])


def plot_effective_df(x: np.ndarray, lambdas: np.ndarray) -> plt.Figure:
    """
    Visualisiert die effektiven Freiheitsgrade als Funktion von log(λ).
    :param x: standardisierte Merkmalsmatrix der Form `(n, p)`.
    :param lambdas: 1D-Array mit Regularisierungsstärken.
    :return: das `Figure`-Objekt.
    """
    df = effective_df_ridge(x, lambdas)
    txt = f'OLS-Grenzwert df = p = {x.shape[1]}'

    figure, ax = plt.subplots(figsize=(7, 3.5), dpi=100)
    xs = np.log10(lambdas)
    ax.plot(xs, df, color='tab:purple', linewidth=2)
    ax.axhline(x.shape[1], color='red', linestyle='--', label=txt)
    ax.axhline(0, color='black', linewidth=0.5)
    ax.set(axisbelow=True, xlabel=r'$\log_{10}(\lambda)$', xlim=[xs[0], xs[-1]])
    ax.set_ylabel(r'Effektive Freiheitsgrade $\mathrm{df}(\lambda)$')
    ax.grid(color='#A0A0A0', linestyle='--', linewidth=0.5)
    ax.legend()
    figure.tight_layout()
    return figure


figure = plot_effective_df(x_train_poly, lambdas_ridge)
plt.show(figure)

&nbsp;Mit $\lambda$ haben wir eine **kontinuierliche Steuerung** der Modellkomplexität. Während Best Subset diskret von $k = 7$ auf $k = 8$ springt, gleitet Ridge stufenlos von 44 Freiheitsgraden hinunter zu 0. Das macht Ridge-Modelle weniger anfällig für Varianzen in der Modellwahl.

### Schrumpfungsfaktor pro Hauptkomponente

Aus der SVD-Analyse folgt eine wunderschöne Aussage: Ridge schrumpft die Koeffizienten **in der Hauptkomponenten-Basis** unabhängig voneinander. Konkret: schreibt man $\hat{\boldsymbol\beta}_{\text{Ridge}}$ in der Basis der Rechts-Singulärvektoren $\mathbf{v}_j$, dann gilt

$$
\hat{\boldsymbol\beta}_{\text{Ridge}} = \sum_{j=1}^{p} \underbrace{\frac{d_j^2}{d_j^2 + \lambda}}_{\text{Schrumpfungsfaktor}}
\cdot \frac{\mathbf{u}_j^\top \mathbf{y}}{d_j} \cdot \mathbf{v}_j.
$$

Der OLS-Beitrag der $j$-ten Komponente ist $\frac{\mathbf{u}_j^\top \mathbf{y}}{d_j} \cdot \mathbf{v}_j$; Ridge multipliziert ihn mit $\frac{d_j^2}{d_j^2 + \lambda} \in (0, 1)$.

**Wichtige Konsequenz:**

- Komponenten mit **grossem** $d_j$ (hochvariante Richtungen in $\mathbf{X}$) werden **wenig geschrumpft**.
- Komponenten mit **kleinem** $d_j$ (rauscharme Richtungen) werden **stark geschrumpft**.

Das ist genau das, was wir wollen: rauschhafte Richtungen werden gedämpft, signalreiche Richtungen bleiben erhalten.


In [ ]:
def plot_pc_shrinkage(x: np.ndarray, lambdas: np.ndarray, show_first_k: int) -> plt.Figure:
    """
    Visualisiert den Schrumpfungsfaktor pro Hauptkomponente als Funktion von log(λ).
    :param X: standardisierte Designmatrix der Form `(n, p)`.
    :param lambdas: 1D-Array mit Regularisierungsstärken.
    :param show_first_k: Anzahl der zu zeichnenden Hauptkomponenten (von gross nach klein).
    :return: das `Figure`-Objekt.
    """
    s = np.linalg.svd(x, compute_uv=False)
    d2 = s ** 2
    k = min(show_first_k, d2.size)

    figure, ax = plt.subplots(figsize=(8, 4), dpi=100)
    xs = np.log10(lambdas)
    cmap = plt.get_cmap('viridis')
    for j in range(k):
        shrinkage = d2[j] / (d2[j] + lambdas)
        color = cmap(j / max(k - 1, 1))
        txt = f'PK {j+1}  ($d_j^2 = {d2[j]:.2f}$)'
        ax.plot(xs, shrinkage, color=color, linewidth=1.7, label=txt)
    ax.axhline(1.0, color='black', linewidth=0.5, linestyle=':')
    ax.axhline(0.0, color='black', linewidth=0.5, linestyle=':')
    ax.set(xlabel=r'$\log_{10}(\lambda)$', xlim=[xs[0], xs[-1]])
    ax.set(axisbelow=True, ylabel=r'Schrumpfungsfaktor $\frac{d_j^2}{d_j^2 + \lambda}$')
    ax.grid(color='#A0A0A0', linestyle='--', linewidth=0.5)
    ax.legend(fontsize=8, ncol=2, loc="best")
    figure.tight_layout()
    return figure


figure = plot_pc_shrinkage(x_train_poly, lambdas_ridge, show_first_k=10)
plt.show(figure)

**Lesehilfe für den Plot:** Jede Linie zeigt, wie stark die $j$-te Hauptkomponente (PK) erhalten bleibt. Die *gelbe* Linie (PK mit dem kleinsten $d_j^2$) fällt schon bei moderatem $\lambda$ in Richtung 0; die *dunkelblauen* Linien (grosse $d_j^2$) bleiben lange nahe bei 1. So filtert Ridge automatisch das Rauschen heraus, ohne dass wir selber entscheiden müssen, welche Komponenten wichtig sind.


## Lasso (L1-Regularisierung)

### Mathematische Formulierung

Lasso minimiert die Verlustfunktion mit einem L1-Strafterm:

$$
\hat{\boldsymbol\beta}_{\text{Lasso}}
\;=\; \arg\min_{\boldsymbol\beta}
\Big\{ \|\mathbf{y} - \mathbf{X}\boldsymbol\beta\|_2^2
       \;+\; \lambda \, \|\boldsymbol\beta\|_1 \Big\},
\qquad \|\boldsymbol\beta\|_1 = \sum_{j=1}^{p} |\beta_j|.
$$

**Drei wichtige Unterschiede zu Ridge:**

1. **Keine geschlossene Lösung.** Der L1-Term ist an $\beta_j = 0$ nicht differenzierbar; man verwendet z. B. *Coordinate Descent* (was [`sklearn.Lasso`](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Lasso.html) intern macht).
2. **Sparse Lösungen.** Bei genügend grossem $\lambda$ werden Koeffizienten **exakt null** &mdash; Lasso macht also implizit Variablenselektion.
3. **Stückweise lineare Pfade.** Im Gegensatz zu den glatten Ridge-Pfaden sind Lasso-Pfade aus Geradenstücken zusammengesetzt; an den Knicken tritt eine Variable in das Modell ein oder aus.

> **`sklearn`-Detail:** Die Verlustfunktion in `sklearn.Lasso` enthält zusätzlich einen Vorfaktor $\frac{1}{2n}$:
> $$\frac{1}{2n}\|\mathbf{y} - \mathbf{X}\boldsymbol\beta\|_2^2 + \alpha\,\|\boldsymbol\beta\|_1.$$
> Das hier verwendete `alpha` ist also nicht *exakt* das $\lambda$ aus der Vorlesungsformel oben &mdash; aber das Verhalten ("grösseres `alpha` ⇒ mehr Schrumpfung") ist identisch.


### Übung: Lasso-Variablenselektion

**Ihre Aufgabe:**

1. Fitten Sie ein [Lasso](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Lasso.html)-Modell mit `alpha = 0.05` auf die polynomialen Merkmale. Beschränken Sie die maximale Zahl an Iterationen auf $50000$.
2. Zählen Sie, wie viele Koeffizienten exakt Null sind. Welche Merkmale haben es in das Modell geschafft?
3. Berechnen Sie den Koeffizientenpfad für eine Sequenz von `alpha`-Werten (verwenden Sie `compute_coefficient_path` mit `method='lasso'`) und stellen Sie ihn neben dem Ridge-Pfad dar.


In [ ]:
# 1. Lasso-Modell mit alpha = 0.05 trainieren
lasso = Lasso(alpha=0.05, max_iter=50000)
lasso.fit(x_train_poly, y_train)
print(f'    Form vom Gewichtsvektor: {lasso.coef_.shape}')
print(f'Datentyp vom Gewichtsvektor: {lasso.coef_.dtype}')
print()

# 2. Anzahl Null-Koeffizienten zählen
epsilon = 1e-12
is_retained = np.abs(lasso.coef_) >= epsilon
zero_count = np.sum(~is_retained).item()
nonzero_count = lasso.coef_.size - zero_count
print(f'      Anzahl Null-Koeffizienten: {zero_count}')
print(f'Anzahl Nicht-Null-Koeffizienten: {nonzero_count}')
print(f'Gebliebene Merkmale:')
for fn, retained in zip(feature_names_poly, is_retained):
    if retained:
        print(f'  {fn}')

In [ ]:
# 3. Koeffizientenpfad plotten und mit dem Ridge-Pfad vergleichen
lambdas_lasso = np.logspace(-4, 1, 126)
coefs_lasso =\
    compute_coefficient_path(x_train_poly, y_train, lambdas_lasso, method='lasso')
figure = plot_coefficient_path(lambdas_lasso, coefs_lasso, feature_names_poly)
plt.show(figure)

**Beobachtungen:**
- Während der Gesamteffekt der Erhöhung von $\lambda$ darin besteht, dass die Koeffizienten in Richtung Null schrumpfen, können einzelne Koeffizienten während dieses Prozesses gelegentlich ansteigen.
- Lasso-Pfade laufen *spitz* in die Null hinein; Ridge-Pfade nähern sich der Null nur asymptotisch.

### Anzahl aktiver Merkmale entlang des Lasso-Pfads

Bei Lasso interpretieren wir die Anzahl von Nicht-Null-Koeffizienten als "Modellgröße". So können die Tendenz beobachten, dass Lasso mit zunehmendem $\lambda$ Merkmale **entfernt**.

In [ ]:
def plot_n_active_features(lambdas: np.ndarray, coefs_matrix: np.ndarray) -> plt.Figure:
    """
    Visualisiert die Anzahl aktiver (Nicht-Null-) Merkmale entlang eines Pfads.
    :param lambdas: 1D-Array mit Regularisierungsstärken.
    :param coefs_matrix: Matrix der Form `(len(lambdas), p)`.
    :return: das neu erstellte `Figure`-Objekt.
    """
    n_active = np.sum(coefs_matrix != 0, axis=1)
    xs = np.log10(lambdas)
    figure, ax = plt.subplots(figsize=(7, 3.5), dpi=100)
    ax.plot(xs, n_active, '-', color='tab:green', markersize=3)
    ax.set(axisbelow=True, xlabel=r'$\log_{10}(\lambda)$', xlim=[xs[0], xs[-1]])
    ax.set_ylabel('Anzahl Nicht-Null-Koeffizienten')
    ax.grid(color='#A0A0A0', linestyle='--', linewidth=0.5)
    figure.tight_layout()
    return figure


figure = plot_n_active_features(lambdas_lasso, coefs_lasso)
plt.show(figure)

## Elastic Net

Elastic Net kombiniert L1 und L2:

$$
\hat{\boldsymbol\beta}_{\text{EN}}
\;=\; \arg\min_{\boldsymbol\beta}
\Big\{ \|\mathbf{y} - \mathbf{X}\boldsymbol\beta\|_2^2
\;+\; \lambda \Big[\, \alpha\,\|\boldsymbol\beta\|_1 + (1-\alpha)\,\|\boldsymbol\beta\|_2^2 \Big] \Big\}.
$$

Mit Mischungsparameter $\alpha \in [0, 1]$:

- $\alpha = 0$ &nbsp;&rArr;&nbsp; Ridge.
- $\alpha = 1$ &nbsp;&rArr;&nbsp; Lasso.
- $\alpha \in (0, 1)$ &nbsp;&rArr;&nbsp; Mischung &mdash; behält Lasso-artige Variablenselektion, ist aber bei stark **korrelierten Merkmalen** stabiler (Lasso wählt oft *zufällig* eine aus einer Gruppe korrelierter Merkmale; Elastic Net behält tendenziell alle, aber geschrumpft).

> **`sklearn`-Stolperstein:** In [`sklearn.ElasticNet`](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.ElasticNet.html) heisst der Mischungsparameter `l1_ratio`. Es gilt:
> $$\text{`alpha`}_{\text{sklearn}} = \lambda, \qquad \text{`l1\_ratio`}_{\text{sklearn}} = \alpha_{\text{Vorlesung}}.$$
> Der `sklearn`-Verlust lautet konkret:
> $$\frac{1}{2n}\|\mathbf{y} - \mathbf{X}\boldsymbol\beta\|_2^2 \;+\; \text{alpha}\cdot\Big[\, \text{l1\_ratio}\cdot\|\boldsymbol\beta\|_1 + \tfrac{1}{2}(1-\text{l1\_ratio})\cdot\|\boldsymbol\beta\|_2^2 \Big].$$


### Übung

1. Trainieren Sie Elastic-Net-Modelle auf dem Trainingsdatensatz. Fixieren Sie `alpha = 0.05` und variieren Sie `l1_ratio` über die Werte `[0.001, 0.25, 0.5, 0.75, 1.0]`.
2. Für jeden Wert: zählen Sie die Nicht-Null-Koeffizienten und berechnen Sie den Test-MSE.
3. Erstellen Sie einen Plot: Anzahl aktiver Merkmale (x-Achse) gegen Test-MSE (y-Achse), beschriftet mit dem jeweiligen `l1_ratio`-Wert.

**Hinweis:** Bei `l1_ratio = 0` gibt `sklearn` eine Warnung aus, dass man besser direkt `Ridge` nutzen sollte. Deswegen ist die erste Zahl in der Sequenz oben `0.001`.

In [ ]:
l1_ratios = [0.001, 0.25, 0.5, 0.75, 1.0]
alpha = 0.05
epsilon = 1e-11

# Für jedes l1_ratio: Modell fitten, n_aktiv und Test-MSE notieren
active_features_count = np.empty(len(l1_ratios), dtype=np.uint16)
mses = np.empty((len(l1_ratios), 2), dtype=np.float64)  # (ratios, [train, test])
for i, ratio in enumerate(l1_ratios):
    en = ElasticNet(alpha=alpha, l1_ratio=ratio, max_iter=100000)
    en.fit(x_train_poly, y_train)
    active_features_count[i] = np.sum(np.abs(en.coef_) > epsilon)
    mses[i, :] = evaluate(en, x_train_poly, x_test_poly, y_train, y_test)

# Plot erstellen
figure, ax = plt.subplots(figsize=(8, 4), dpi=100)
ax.plot(active_features_count, mses[:, 0], 'o--', color='tab:green', label='Train-MSE')
ax.plot(active_features_count, mses[:, 1], 'o--', color='tab:purple', label='Test-MSE')
for position_x, position_y, ratio in zip(active_features_count, mses[:, 1], l1_ratios):
    xy = (position_x, position_y)
    xy_text = (position_x - 1, position_y - 0.03)
    ax.annotate(f'ratio = {ratio}', xy=xy, xytext=xy_text)
ax.set(axisbelow=True, xlabel='Anzahl aktiver Merkmale', ylabel='MSE')
ax.grid(color='#A0A0A0', linestyle='--', linewidth=0.5)
ax.legend()
figure.tight_layout()
plt.show(figure)

**Beobachtung:** Mit zunehmendem `l1_ratio` werden mehr Koeffizienten genau Null. Der Test-MSE kann sich dabei in die eine oder andere Richtung bewegen &mdash; Elastic Net ist ein Werkzeug, das wir per Kreuzvalidierung "feintunen" sollten.


## Hyperparameter wählen mit Kreuzvalidierung

Bislang haben wir $\lambda$ und `l1_ratio` von Hand gesetzt. In der Praxis wählen wir sie per **Kreuzvalidierung**. `sklearn` bietet dafür drei spezialisierte Klassen:

- [`RidgeCV(alphas=...)`](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.RidgeCV.html) für Ridge.
- [`LassoCV(alphas=...)`](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LassoCV.html) für Lasso.
- [`ElasticNetCV(alphas=..., l1_ratio=...)`](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.ElasticNetCV.html) für Elastic Net.

Die `*CV`-Klassen sind effizienter als ein vollständiges [`GridSearchCV`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html), weil sie die Pfade des Algorithmus ausnutzen.

### Übung

Im Quellcode unten werden `RidgeCV`, `LassoCV` und `ElasticNetCV` jeweils mit $5$-facher Kreuzvalidierung trainiert.

Erweitern Sie den Code-Ausschnitt, indem Sie den für jede Methode ausgewählten optimalen Hyperparameter ausgeben.

In [ ]:
alphas = np.logspace(-3, 2, 50)
l1_ratios = [0.001, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 0.999]
fold_splitter = KFold(n_splits=5, shuffle=True, random_state=SEED)

# RidgeCV
ridge_cv = RidgeCV(alphas=alphas, cv=fold_splitter)
ridge_cv.fit(x_train_poly, y_train)

# LassoCV
lasso_cv = LassoCV(alphas=alphas, cv=fold_splitter, max_iter=100000, random_state=SEED)
lasso_cv.fit(x_train_poly, y_train)

# ElasticNetCV
en_cv = ElasticNetCV(alphas=alphas, l1_ratio=l1_ratios,
                     cv=fold_splitter, max_iter=100000, random_state=SEED)
en_cv.fit(x_train_poly, y_train)

In [ ]:
# Optimale Parameterwerte berichten
def count_active(model) -> int:
    return (np.abs(model.coef_) > 1e-11).sum().item()

print('   Methode  |   Alpha  | L1 Ratio | Aktive Merkmale ')
print('------------+----------+----------+-----------------')
print(f'     Ridge  |  {ridge_cv.alpha_:3.4f} |          | {count_active(ridge_cv)}')
print(f'     Lasso  |   {lasso_cv.alpha_:3.4f} |          | {count_active(lasso_cv)}')
print(f'Elastic Net |   {en_cv.alpha_:3.4f} |   {en_cv.l1_ratio_:.4f} | {count_active(en_cv)}')

Die optimalen Parameter werden natürlich auf der Grundlage der CV-Fehlerschätzungen ausgewählt. Die `*CV`-Objekte speichern diese Validierungsfehler pro Fold, sodass wir sie z.B. grafisch darstellen können. Der folgende Codeausschnitt zeigt dies für das Objekt `lasso_cv`.

In [ ]:
def plot_lasso_cv_curve(model) -> plt.Figure:
    """
    Visualisiert die CV-Fehlerkurve eines gefitteten LassoCV-Modells.
    :param model: gefittetes `LassoCV`-Objekt mit `mse_path_` und `alphas_`.
    :return: das neu erstellte `Figure`-Objekt.
    """
    # mse_path_ hat die Form (alphas, folds)
    mse_mean = model.mse_path_.mean(axis=1)
    mse_std = model.mse_path_.std(axis=1)
    xs = np.log10(model.alphas_)
    alpha_best = model.alpha_

    figure, ax = plt.subplots(figsize=(8, 3.5), dpi=100)
    ax.plot(xs, mse_mean, color='tab:purple', linewidth=2, label='Mittlerer CV-MSE')
    ax.fill_between(xs, mse_mean - mse_std, mse_mean + mse_std,
                    color='tab:purple', alpha=0.2,
                    label=r'$\pm 1$ Standardabweichung')
    ax.axvline(np.log10(alpha_best), color='black', linestyle='--',
               label=f'optimales $\\alpha$ = {alpha_best:.4f}')
    ax.set(axisbelow=True, xlim=[xs[0], xs[-1]])
    ax.set(xlabel=r'$\log_{10}(\alpha)$', ylabel='CV-MSE')
    ax.grid(color='#A0A0A0', linestyle='--', linewidth=0.5)
    ax.legend()
    figure.tight_layout()
    return figure


fig = plot_lasso_cv_curve(lasso_cv)
plt.show()

In diesem Notebook haben wir mehrere Modelle für die Vorhersage des log(PSA)-Wertes ausprobiert. Hier fassen wir alle Ergebnisse zusammen.

In [ ]:
methods = {
    'OLS 8 Merkmale': ols_baseline,
    'OLS 44 Merkmale': ols_poly,
    'Ridge': ridge_cv,
    'Lasso': lasso_cv,
    'Elastic Net': en_cv
}

# Train- und Test-MSE für alle Modelle berechnen
mses = np.empty((len(methods), 2), dtype=np.float64)
for i, model in enumerate(methods.values()):
    if model.coef_.size == 8:
        x_train_current = x_train
        x_test_current = x_test
    else:
        x_train_current = x_train_poly
        x_test_current = x_test_poly
    mses[i, :] = evaluate(model, x_train_current, x_test_current, y_train, y_test)

# Train- und Test-MSEs vergleichen
figure, ax = plt.subplots(figsize=(4, 4), dpi=100)
xs = np.arange(len(methods))
ax.bar(xs - 0.2, np.sqrt(mses[:, 0]), width=0.4, label='Train-MSE')
ax.bar(xs + 0.2, np.sqrt(mses[:, 1]), width=0.4, label='Test-MSE')
ax.set(axisbelow=True, ylabel='RMSE')
ax.set(xticks=xs)
ax.set_xticklabels(list(methods.keys()), ha='right', rotation=45)
ax.grid(color='#A0A0A0', linestyle='--', linewidth=0.5)
ax.legend()
figure.tight_layout()
plt.show(figure)

**Schlüsselbeobachtung:** Die regularisierten Methoden &mdash; insbesondere Lasso und Elastic Net &mdash; erzielen einen deutlich besseren Test-MSE als das überangepasste OLS-Modell mit 44 Merkmalen, und zusätzlich liefert Lasso eine *interpretierbare* Auswahl von wenigen wirklich relevanten Merkmalen.

## Zusammenfassung: Methoden

| Methode | Strafterm | Schliessliche Lösung | Variablenselektion? | Pfad |
|---|---|---|---|---|
| OLS | &mdash; | $(\mathbf{X}^\top\mathbf{X})^{-1}\mathbf{X}^\top\mathbf{y}$ | nein | trivial |
| **Ridge** | $\lambda\|\beta\|_2^2$ | $(\mathbf{X}^\top\mathbf{X}+\lambda \mathbf{I})^{-1}\mathbf{X}^\top\mathbf{y}$ | nein | glatt |
| **Lasso** | $\lambda\|\beta\|_1$ | iterativ (Coordinate Descent) | **ja** | stückw. linear |
| **Elastic Net** | $\lambda(\alpha\|\beta\|_1 + (1-\alpha)\|\beta\|_2^2)$ | iterativ | **ja**, gruppenfähig | stückw. linear |

**Drei Take-Home-Messages:**

1. **Regularisierung ist nicht optional**, sobald $p$ in der Nähe von $n$ oder grösser ist. Sie reduziert die Varianz Ihrer Schätzer dramatisch.
2. **Standardisierung ist Pflicht** bei Ridge, Lasso, Elastic Net &mdash; sonst werden Merkmale ungerecht behandelt.
3. **Den Hyperparameter wählen Sie per Kreuzvalidierung.** `RidgeCV`, `LassoCV`, `ElasticNetCV` sind dabei hilfreich &mdash; aber denken Sie an die Stolpersteine in der Tabelle unten.

## Klassifikations-Add-on: Logistische Regression mit Strafterm

Regularisierung ist nicht nur für Regressionsprobleme &mdash; sie ist genauso für die Klassifikation relevant. Die Klasse [`LogisticRegression`](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html) in `sklearn` unterstützt L1, L2 und Elastic Net über den Parameter `penalty`.

Hier benutzen wir denselben Prostate-Datensatz, definieren wir aber eine *binäre Zielvariable*, indem wir den `lpsa`-Median als Schwellenwert nehmen ("hoher" vs. "niedriger" PSA-Wert). So bleiben wir bei den vertrauten Merkmalen und brauchen keinen neuen Datensatz einzuführen.

**Wichtige Punkte:**

- `LogisticRegression(C=1.0)` &mdash; L2-Regularisierung (Default).
- `LogisticRegression(C=1.0, l1_ratio=1, solver='liblinear')` &mdash; L1, benötigt einen kompatiblen Solver.
- `LogisticRegression(C=1.0, l1_ratio=0.5, solver='saga')` &mdash; Elastic Net, benötigt `saga`-Solver.
- $C = 1/\lambda$ &mdash; **grösseres `C` = schwächere Regularisierung**.

> **Hinweis zur Version:** Die alte Syntax mit dem Parameter `penalty` funktioniert noch, ist aber Ab `scikit-learn` 1.8 als *deprecated* markiert.

In [ ]:
# Binäres Ziel: lpsa über Median = "hoch" (1), sonst "niedrig" (0)
median_lpsa = np.median(y_train)
y_train_binary = (y_train > median_lpsa).astype(np.uint16)
y_test_binary = (y_test > median_lpsa).astype(np.uint16)

# Wir benutzen weiter die polynomial erweiterten Merkmale
print(f'Anteil Klasse 1 im Training: {y_train_binary.mean() * 100:.1f}%')
print(f'    Anteil Klasse 1 im Test: {y_test_binary.mean() * 100:.1f}%')

In [ ]:
from sklearn.linear_model import LogisticRegression, LogisticRegressionCV
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score

# Drei Varianten der logistischen Regression
log_l2 = LogisticRegression(C=1.0, l1_ratio=0, solver='lbfgs',
                            max_iter=10000, random_state=SEED)
log_l1 = LogisticRegression(C=1.0, l1_ratio=1, solver='liblinear',
                            max_iter=10000, random_state=SEED)
log_en = LogisticRegression(C=1.0, l1_ratio=0.5, solver='saga',
                            max_iter=10000, random_state=SEED)

for model_name, model in [('L2', log_l2), ('L1', log_l1), ('Elastic Net', log_en)]:
    model.fit(x_train_poly, y_train_binary)
    train_accuracy = accuracy_score(y_train_binary, model.predict(x_train_poly))
    test_accuracy = accuracy_score(y_test_binary, model.predict(x_test_poly))
    print(f'                 Modell: {model_name}')
    print(f'Anzahl aktiver Merkmale: {np.sum(np.abs(model.coef_) > 1e-10)}')
    print(f'   Training Genauigkeit: {train_accuracy * 100:.1f}%')
    print(f'       Test Genauigkeit: {test_accuracy* 100:.1f}%')
    print()

**Beobachtung:** Genau wie im Regressionsfall führt L1 zu **sparseren** Lösungen, während L2 alle Merkmale behält. Bei diesem kleinen Datensatz sind die Genauigkeiten ähnlich, aber das **Interpretationspotenzial** unterscheidet sich erheblich.

> **Profi-Tipp:** Für die automatische Hyperparameterwahl gibt es `LogisticRegressionCV`. Genauso wie `RidgeCV` oder `LassoCV` macht sie CV intern. Auch hier ist `Cs` der Hyperparameter (Liste oder Anzahl logarithmisch verteilter Werte zwischen `1e-4` und `1e4`).


## Zusammenfassung: `sklearn`-Stolpersteine

| Stolperstein | Erklärung | Workaround |
|---|---|---|
| **`alpha` ≠ $\lambda$ bei Lasso und EN** | `sklearn` skaliert den Datenterm mit $\frac{1}{2n}$. Verhalten ist qualitativ identisch, der numerische Wert anders. | Akzeptieren &mdash; Sie wählen `alpha` ja sowieso per CV. |
| **`alpha` bei `ElasticNet`** vs. **`alpha`** in der Vorlesung | `sklearn`-`alpha` ist das *Gesamt-$\lambda$*. Der Vorlesungs-`alpha` (Mischung) heisst hier `l1_ratio`. | Schreiben Sie sich die Zuordnung an die Tafel. |
| **`l1_ratio=0` in `ElasticNet`** | Wirft Konvergenzwarnung. | `Ridge` direkt verwenden, oder `l1_ratio=1e-6`. |
| **`RidgeCV` macht GCV, nicht KFold** | Standard `cv=None` ⇒ GCV (eine LOO-Approximation). | Explizit `cv=KFold(...)` setzen. |
| **`normalize=True` ist deprecated** | Wurde 2021 entfernt. | `Pipeline([("scaler", StandardScaler()), ("model", Ridge())])`. |
| **`LogisticRegression(C=...)`** | `C = 1/λ` &mdash; *invertiert*. Grösseres `C` ⇒ *weniger* Regularisierung. | "C wie Confidence-in-data": viel C, wenig Strafterm. |
| **`Lasso` braucht `solver`-Wahl bei logistischer Regression** | `LogisticRegression(penalty='l1')` funktioniert nur mit `solver='liblinear'` oder `solver='saga'`. | Beim Setzen der Strafe immer auch `solver` setzen. |
